In [1]:
import pandas as pd

import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv('drone_communication_dataset.csv')


In [3]:
df["label_gps_spoofing"].value_counts()


label_gps_spoofing
0    43434
1     1855
Name: count, dtype: int64

In [4]:
df.head()

,timestamp,signal_strength,packet_loss_rate,round_trip_time,communication_protocol,frequency_band,encryption_type,drone_gps_coordinates,altitude,speed_trajectory,...,intrusion_detection_flags,temporal_patterns,label_normal,label_spoofing,label_mitm,label_ddos,label_gps_spoofing,label_malware,label_jamming,label_protocol_exploit
0,2019-11-01 00:00:00,-60,0,50,ZigBee,5.0,AES,"(-11.69864152850468, 86.89778588010734)",291,60,...,0,0,0,0,0,0,0,0,0,0
1,2019-11-01 01:00:00,-64,0,50,LoRa,2.4,AES,"(66.98918792506822, -140.91891468822496)",100,48,...,0,0,0,0,0,0,0,0,0,0
2,2019-11-01 02:00:00,-60,0,194,ZigBee,2.4,AES,"(-31.711586784829194, 36.5632095665857)",226,30,...,0,0,0,0,0,0,0,0,0,0
3,2019-11-01 03:00:00,-60,0,50,Wi-Fi,2.4,AES,"(3.5943733073538198, 74.7992608739666)",100,30,...,0,0,1,0,0,0,0,0,1,0
4,2019-11-01 04:00:00,-69,0,50,Wi-Fi,2.4,Plain-text,"(41.4738797417983, -100.00817301602393)",100,30,...,0,0,0,0,0,0,0,0,0,0


In [5]:
df_filtered = df[(df['label_normal'] == 1) | (df['label_gps_spoofing'] == 1)].copy()

# 3. Zdefiniowanie zmiennej docelowej y
# 1 oznacza atak GPS Spoofing, 0 oznacza ruch normalny (bo odfiltrowaliśmy inne)
y = df_filtered['label_gps_spoofing']

# 4. Czyszczenie zbioru X (Usuwamy wycieki, identyfikatory i szum)
cols_to_drop = [
    # A. Wszystkie labele (bez nich model miałby gotowe odpowiedzi)
    'label_normal', 'label_spoofing', 'label_mitm', 'label_ddos', 
    'label_gps_spoofing', 'label_malware', 'label_jamming', 'label_protocol_exploit',
    
    # B. Wycieki danych (sprzętowe flagi bezpieczeństwa)
    'gps_signal_integrity', 'intrusion_detection_flags', 
    'malware_detection_signals', 'anomaly_in_behavioral_pattern',
    
    # C. Czas i ID (aby uniknąć uczenia się godzin i nazw)
    'timestamp', 'drone_identification',
    
    # D. Tymczasowo usuwamy kolumny o formacie tekstowym/współrzędnych. 
    # Będziesz mógł je dodać później w fazie Feature Engineering po sparsowaniu.
    'drone_gps_coordinates', 'speed_trajectory', 'temporal_patterns'
]

# Bezpieczne usunięcie kolumn (tylko tych, które faktycznie istnieją w dataframe)
cols_to_drop = [c for c in cols_to_drop if c in df_filtered.columns]
X = df_filtered.drop(columns=cols_to_drop)

In [6]:
from sklearn.preprocessing import LabelEncoder
import sklearn as sk

le = LabelEncoder()
y = le.fit_transform(y)

X_train, X_test, y_train, y_test = sk.model_selection.train_test_split(
    X, y, test_size=0.2, shuffle=True, random_state=42
)

In [7]:
X.describe()

,signal_strength,packet_loss_rate,round_trip_time,frequency_band,altitude,transmission_power,message_authentication_status,session_key_validity,signal_noise_ratio,sequence_number_gap,data_rate,network_traffic_volume,uplink_downlink_quality,base_station_load,port_scanning_attempts,drone_signal_handoff
count,6284.000000,6284.000000,6284.000000,6284.000000,6284.000000,6284.000000,6284.000000,6284.000000,6284.000000,6284.000000,6284.000000,6284.000000,6284.000000,6284.000000,6284.000000,6284.0
mean,-60.368555,2.590388,100.533896,183.255983,114.548695,20.795990,0.898472,0.847072,29.704488,0.250796,126.195735,395.275780,51.611871,70.207034,0.092934,1.0
std,5.247138,8.919357,113.785120,362.843320,61.823348,5.034031,0.302050,0.359947,4.200495,1.221117,80.440835,551.360218,9.815409,54.073770,0.762370,0.0
min,-85.000000,0.000000,50.000000,2.400000,21.000000,5.000000,0.000000,0.000000,5.000000,0.000000,50.000000,100.000000,20.000000,50.000000,0.000000,1.0
25%,-60.000000,0.000000,50.000000,2.400000,100.000000,20.000000,1.000000,1.000000,30.000000,0.000000,100.000000,100.000000,50.000000,50.000000,0.000000,1.0
50%,-60.000000,0.000000,50.000000,2.400000,100.000000,20.000000,1.000000,1.000000,30.000000,0.000000,100.000000,100.000000,50.000000,50.000000,0.000000,1.0
75%,-60.000000,0.000000,50.000000,5.000000,100.000000,20.000000,1.000000,1.000000,30.000000,0.000000,100.000000,520.250000,50.000000,50.000000,0.000000,1.0
max,-41.000000,49.000000,499.000000,915.000000,499.000000,49.000000,1.000000,1.000000,49.000000,9.000000,499.000000,1999.000000,99.000000,299.000000,9.000000,1.0


In [8]:
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

num_cols = X.select_dtypes(include=['number']).columns
cat_cols = X.select_dtypes(include=['object']).columns

preprocessor = ColumnTransformer(
    transformers=[
        ('num', MinMaxScaler(), num_cols),
        ('cat', OneHotEncoder(drop='if_binary', handle_unknown='ignore'), cat_cols)
    ]
)
X_train = preprocessor.fit_transform(X_train)

X_test = preprocessor.transform(X_test)

In [9]:
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

models = {
    'DT' : DecisionTreeClassifier(),
    'RF' : RandomForestClassifier(),
    'XGBoost' : XGBClassifier(eval_metric='mlogloss'),
    'CatBoost' : CatBoostClassifier(verbose=0),
    'LightGBM' : LGBMClassifier(verbose=-1),
}

In [10]:
from sklearn.metrics import accuracy_score

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print(name + ": " + str(accuracy_score(y_test, y_pred)))

DT: 0.5926809864757359
RF: 0.649164677804296
XGBoost: 0.6587112171837709
CatBoost: 0.6754176610978521
LightGBM: 0.6682577565632458


In [11]:
# Sprawdzenie, co zepsuło wynik (Feature Importance)
import matplotlib.pyplot as plt

xgb_model = models['XGBoost']
importances = pd.Series(xgb_model.feature_importances_, index=preprocessor.get_feature_names_out())

# Wyświetl 10 najbardziej "podejrzanych" kolumn
print(importances.sort_values(ascending=False).head(10))

num__message_authentication_status    0.058666
num__base_station_load                0.058332
num__uplink_downlink_quality          0.057727
num__network_traffic_volume           0.054862
num__altitude                         0.054091
num__round_trip_time                  0.051979
num__signal_strength                  0.051301
num__packet_loss_rate                 0.049832
cat__communication_protocol_ZigBee    0.048931
num__transmission_power               0.047012
dtype: float32
